# 26 AG census-division readouts

**Purpose.** To see where in the country the ledger's gains and losses fall, this notebook splits the
ledger entries that have a place into the nine census divisions (protocol §3 Geography; amendment
A16). It uses the same pairs as notebooks 23 and 24: the six stage-1 pairs and the 308 primary stage-2
pairs.

| Entry by division | Source (`<batch>/derived_cendiv.parquet`, notebook 20 §5) |
|---|---|
| electricity consumers' market payments: −Δ(P_E − X + P_C + P_R) | `P_E`, `X_can`, `P_C`, `P_R` at the load zone |
| each owner group's surplus (§4.4–4.5, A6) | `rev|…`, `tb|…`, `P_H2_*` |
| resource cost (−Δ, so + = saving) | `sc|inv`, `sc|op` |
| −Δ objective cost of power-sector gas (readout) | `gascost_cendiv` |
| emissions and their health damages | `emit_r` by zone; CACES rate of the emitting zone |

The support, taxpayers' credits, the network, gas producers' sales, the foreign actors and both views
stay national (A16). Under the cap, the allowance rows of A14–A15 stay national too; this notebook
reports owners under free allocation only.

**What it writes** (`results/exports/26/`): `cd_pv.parquet`, `cd_check.csv`, `cd_emissions.parquet`,
`cd_stage1.csv`, `cd_stage2_summary.csv`. It opens no ReEDS outputs file and states no hypothesis.


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import time

import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths
from aglib import ledger as L
from aglib import reduce as agr
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("26")
E = L.EXTRACTS_AG
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
CRF = L.crf_after_2026()
EXT = L.load_external()
ZCD = agr.zone_cendiv()
DIVS = sorted(ZCD.unique())
REF = dict(horizon="A", rate="2.0%")
print(len(DIVS), "census divisions:", DIVS)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


9 census divisions: ['East_North_Central', 'East_South_Central', 'Mid_Atlantic', 'Mountain', 'New_England', 'Pacific', 'South_Atlantic', 'West_North_Central', 'West_South_Central']


## 1. The pairs

In [2]:
s1 = pd.DataFrame(dict(stage="1", batch="smr100", case=[f"smr100_{t}_p50" for t in TARGETS], target=TARGETS,
                       condition="reference", base_batch="itcfb", baseline="fbR_none", primary=True, mandate=True,
                       difference="programme"))
P2 = pd.read_csv(paths.exports("24") / "ag2_pairs.csv")
PAIRS = pd.concat([s1, P2[P2["primary"]]], ignore_index=True)
assert len(PAIRS) == 6 + 60 + 178 + 30 + 40
print(PAIRS.groupby("stage").size())

stage
1       6
2a     60
2b    178
2c     30
2d     40
dtype: int64


## 2. Division accounts of one run

`cd_accounts` returns the long table `cendiv, actor, entry, kind, t, Value` (2004 $ per year at each
solve year from 2026). `kind` is `flow` or `lump`, as in the ledger: capital and ITC rows are lumps.
The fuel of gas-curve technologies is scaled by the run's national A6 factor p × Q_power ÷ objective gas
cost, as in `ledger.accounts`.

In [3]:
_D = {}
def derived(batch):
    if batch not in _D:
        d = pd.read_parquet(E / batch / "derived_cendiv.parquet")
        d["t"] = d["t"].astype(int)
        _D[batch] = d
    return _D[batch]


def cd_accounts(batch, case):
    d = derived(batch)
    d = d[(d["case"] == case) & (d["t"] >= 2026)]
    years = np.array(sorted(L._sel(batch, "pvf_onm", case)["t"]))
    years = years[years >= 2026]
    p = L._by_t(L._sel(batch, "repgasprice_nat", case), years)
    q = L._by_t(L._sel(batch, "repgasquant_nat", case), years) * L.QUAD_MMBTU
    g_obj = L._by_t(L._sel(batch, "gascost_cendiv", case), years)
    scale = (p * q / g_obj.where(g_obj != 0)).fillna(1.0)
    w = d.pivot_table(index=["cendiv", "t"], columns="item", values="Value", aggfunc="sum").fillna(0.0)
    w = w.reindex(pd.MultiIndex.from_product([DIVS, years], names=["cendiv", "t"])).fillna(0.0)
    col = lambda k: w[k] if k in w else pd.Series(0.0, index=w.index)
    sc_t = pd.Series(w.index.get_level_values("t").map(scale).values, index=w.index)
    ent = {("electricity consumers", "market payments", "flow"): -(col("P_E") - col("X_can") + col("P_C") + col("P_R")),
           ("system", "resource cost: capital", "lump"): -col("sc|inv"),
           ("system", "resource cost: operating", "flow"): -col("sc|op")}
    for g in L.GROUPS:
        a = L.owner_actor(g)
        ent[(a, "revenue", "flow")] = sum(col(f"rev|{c}|{g}") for c in ("load", "res_marg", "rps", "charge"))
        ent[(a, "credits received: ITC", "lump")] = -col(f"tb|itc|{g}")
        ent[(a, "credits received: other", "flow")] = -col(f"tb|credit|{g}")
        ent[(a, "operating costs", "flow")] = -(col(f"tb|op|{g}") + col(f"tb|fuel_gc|{g}") * sc_t)
        ent[(a, "capital costs", "lump")] = -col(f"tb|inv|{g}")
    ent[("owners: H2 and DAC production", "electricity payments", "flow")] = -(col("P_H2_E") + col("P_H2_C"))
    # the REC trade wedge that the selling generators receive (amendment A23), at the selling state's division
    rc = L._sel(batch, "derived_recs", case)
    rc = rc[rc["item"].str.startswith("wedge_g|") & (rc["t"] >= 2026)]
    part = rc["item"].str.split("|", expand=True)
    rc = rc.assign(o=part[1], g=part[2], cendiv=part[3])
    for g in L.GROUPS:
        for origin, label in L.RPS_WEDGE_ASSIGNED.items():
            x = rc[(rc["o"] == origin) & (rc["g"] == g)]
            ent[(L.owner_actor(g), label, "flow")] = x.groupby(["cendiv", "t"])["Value"].sum().reindex(w.index).fillna(0.0)
    gc = L._sel(batch, "gascost_cendiv", case)
    gc = gc[gc["t"] >= 2026].set_index(["cendiv", "t"])["Value"].reindex(w.index).fillna(0.0)
    ent[("readout", "minus objective gas cost", "flow")] = -gc
    rows = [s.rename("Value").reset_index().assign(actor=a, entry=e, kind=k) for (a, e, k), s in ent.items()]
    return pd.concat(rows, ignore_index=True)[["cendiv", "actor", "entry", "kind", "t", "Value"]], years

**Check against the national ledger.** For the 25 stage-1 and baseline runs, each entry summed over
the nine divisions is compared with the same entry of `ledger.accounts`: consumers' energy, capacity and
RPS payments; each owner group's revenue, credits, operating and capital rows and H2 electricity
payments; and the resource cost.

In [4]:
CHECK_RUNS = [("smr100", c) for c in s1["case"]] + [("itcfb", "fbR_none"), ("itcfb", "fbR_itc30")] + \
             [("batch2", f"bx_{x}_f50") for x in ["aj_p75", "cop28_p50", "cop28_p75", "eia_p75", "eo_p75", "iaea_p50",
                                                  "iaea_p75", "mck_p75"]] + [("ratedesign2", "ax_eia_p95_c60")] + \
             [("batch3", f"bx_{x}_f50_co2cap") for x in ["aj_p75", "cop28_p50", "cop28_p75", "eia_p75", "eo_p75",
                                                         "iaea_p50", "iaea_p75", "mck_p75"]]
NAT_MAP = {"electricity consumers": ["energy payments", "capacity payments", "RPS payments"],
           "system": ["resource cost: capital", "resource cost: operating"]}
rows = []
for b, c in CHECK_RUNS:
    cd, _ = cd_accounts(b, c)
    nat, _m = L.accounts(b, c, EXT)
    for a in ["electricity consumers", "system"] + [L.owner_actor(g) for g in L.GROUPS]:
        x = cd[cd["actor"] == a].groupby("t")["Value"].sum()
        if a in NAT_MAP:
            n = nat[(nat["actor"] == a) & nat["entry"].isin(NAT_MAP[a])]
        else:
            n = nat[nat["actor"] == a]
        n = n.groupby("t")["Value"].sum()
        x, n = x.align(n, fill_value=0.0)
        rows.append(dict(batch=b, case=c, actor=a, max_abs_diff=float((x - n).abs().max()),
                         scale=float(n.abs().max())))
CHK = pd.DataFrame(rows)
CHK["rel"] = CHK["max_abs_diff"] / CHK["scale"].clip(lower=1.0)
CHK.to_csv(OUT / "cd_check.csv", index=False)
display(CHK.groupby("actor")["rel"].max().map(lambda v: f"{v:.1e}"))
assert (CHK["rel"] < 1e-6).all(), "division sums equal the national ledger entries"

actor
electricity consumers                2.9e-08
foreign sellers: Canadian imports    4.0e-08
owners: H2 and DAC production        4.2e-08
owners: SMR (new)                    5.8e-08
owners: coal                         3.9e-08
owners: existing nuclear             8.3e-08
owners: gas                          3.2e-08
owners: hydro                        5.7e-08
owners: other                        5.3e-08
owners: storage                      1.1e-07
owners: wind + solar                 4.4e-08
system                               6.9e-08
Name: rel, dtype: object

## 3. Differences and present values

For each pair: Δ = run − baseline per division and entry, placed on the horizon years with the ledger's
`flow` and `lump` (horizon A: lumps over 30 CRF payments, 2026–2050; horizon B: lumps whole, 2050 flows
held to 2080), then discounted to 2026 at every actor rate (`ledger.RATES`). 2025 $B. Health damages:
Δ process NOx and SO2 of each zone × the CACES rate of that zone, six variants, gain = −damage.

In [5]:
_ACC = {}
def acc(b, c):
    if (b, c) not in _ACC:
        _ACC[(b, c)] = cd_accounts(b, c)
    return _ACC[(b, c)]


MD = EXT["md"]
def health(b, c):
    er = L._sel(b, "emit_r", c)
    er = er[(er["etype"] == "process") & er["eall"].isin(["NOX", "SO2"]) & (er["t"] >= 2026)]
    out = {}
    for v in L.HEALTH:
        m = er.merge(MD[MD["variant"] == v], left_on=["r", "eall"], right_on=["r", "pollutant"], how="inner")
        m["cendiv"] = m["r"].map(ZCD)
        out[v] = (m["Value"] * m["damage"]).groupby([m["cendiv"], m["t"]]).sum()
    return out


t0 = time.time()
recs, emis = [], []
for k, p in PAIRS.iterrows():
    A, ya = acc(p["batch"], p["case"])
    B, yb = acc(p["base_batch"], p["baseline"])
    assert np.array_equal(ya, yb)
    key = ["cendiv", "actor", "entry", "kind", "t"]
    D = A.merge(B, on=key, how="outer", suffixes=("_run", "_base")).fillna(0.0)
    D["Value"] = D["Value_run"] - D["Value_base"]
    ha, hb = health(p["batch"], p["case"]), health(p["base_batch"], p["baseline"])
    for v in L.HEALTH:
        dd = ha[v].sub(hb[v], fill_value=0.0).rename("Value").reset_index()
        D = pd.concat([D, dd.assign(actor="environment", entry=f"health: {v}", kind="flow", Value=-dd["Value"])],
                      ignore_index=True)
    for h in ("A", "B"):
        for cd, g in D.groupby("cendiv"):
            ann = L.annual(g[["actor", "entry", "kind", "t", "Value"]], h, CRF)
            for rk, r in L.RATES.items():
                yrs = L.Y_A if h == "A" else L.Y_B
                f = (1.0 + r) ** -(yrs - 2026) * L.TO2025_FROM_2004 / 1e9
                pvs = ann.values @ f
                recs.append(pd.DataFrame({"cendiv": cd, "actor": ann.index.get_level_values(0),
                                          "entry": ann.index.get_level_values(1), "horizon": h, "rate": rk,
                                          "pv": pvs, "pair": k}))
    em = pd.concat([L._sel(p["batch"], "emit_r", p["case"]).assign(s=1.0),
                    L._sel(p["base_batch"], "emit_r", p["baseline"]).assign(s=-1.0)])
    em = em[em["t"] >= 2026]
    em["cendiv"] = em["r"].map(ZCD)
    e = (em["Value"] * em["s"]).groupby([em["cendiv"], em["etype"], em["eall"], em["t"]]).sum().rename("delta_t").reset_index()
    emis.append(e.assign(pair=k))
    if (k + 1) % 50 == 0:
        print(f"{k + 1} of {len(PAIRS)} pairs, {(time.time() - t0) / 60:.1f} min", flush=True)
meta = PAIRS[["stage", "batch", "case", "target", "condition", "baseline"]]
PV = pd.concat(recs, ignore_index=True).merge(meta, left_on="pair", right_index=True).drop(columns="pair")
PV.to_parquet(OUT / "cd_pv.parquet", index=False)
EM = pd.concat(emis, ignore_index=True).merge(meta, left_on="pair", right_index=True).drop(columns="pair")
EM.to_parquet(OUT / "cd_emissions.parquet", index=False)
print(f"{len(PAIRS)} pairs in {(time.time() - t0) / 60:.1f} min; {len(PV):,} PV rows")

50 of 314 pairs, 0.5 min


100 of 314 pairs, 1.2 min


150 of 314 pairs, 1.8 min


200 of 314 pairs, 2.5 min


250 of 314 pairs, 3.2 min


300 of 314 pairs, 3.8 min


314 pairs in 4.1 min; 2,746,872 PV rows


**Check against notebooks 21 and 24.** For each pair, each actor's PV summed over the divisions
equals its national PV (horizon A and B, 2.0%), for the actors that A16 splits. The national PVs are
taken without the support entries and the allowance entries, which stay national, at lump support and
funding T (the split entries do not depend on those keys).

In [6]:
nat = pd.concat([pd.read_csv(paths.exports("21") / "ag_actor_pv.csv").rename(columns={"run": "case"}).assign(stage="1"),
                 pd.read_parquet(paths.exports("24") / "ag2_actor_pv.parquet")], ignore_index=True)
nat = nat[nat["baseline"].isin(PAIRS["baseline"].unique())]
nat = nat[~nat["entry"].isin(["mandate support", "mandate support charge"]) & ~nat["entry"].str.startswith("allowances")]
nat = nat[nat["support_form"].isin(["lump", "none"]) & nat["funding"].isin(["T", "none"])
          & nat["allowance_national"].isin(["free", "none"]) & nat["allowance_state"].isin(["free", "none"])]
ACT = ["electricity consumers"] + [L.owner_actor(g) for g in L.GROUPS]
sel = PV[PV["actor"].isin(ACT) & (PV["rate"] == "2.0%")]
mine = sel.groupby(["stage", "case", "baseline", "horizon", "actor"])["pv"].sum()
n = nat[nat["actor"].isin(ACT)].groupby(["stage", "case", "baseline", "horizon", "actor"])["PV_2.0%"].sum()
m = pd.concat([mine.rename("cendiv_sum"), n.rename("national")], axis=1, join="inner")
m["diff"] = m["cendiv_sum"] - m["national"]
print(f"{len(m)} pair-horizon-actor cells; largest |diff| {m['diff'].abs().max():.2e} 2025 $B")
assert m["diff"].abs().max() < 1e-3

6908 pair-horizon-actor cells; largest |diff| 5.97e-04 2025 $B


## 4. Stage 1

Reference variant: horizon A, 2.0%, 2025 $B, PV to 2026. Rows are divisions; one table per actor
group. Owners are summed over the owner groups here; the export keeps each group.

In [7]:
s1v = PV[(PV["stage"] == "1") & (PV["horizon"] == REF["horizon"]) & (PV["rate"] == REF["rate"])]
lab = s1v["actor"].where(~s1v["actor"].str.startswith("owners:"), "owners (all groups)")
lab = lab.where(~s1v["entry"].str.startswith("health"), "health: " + s1v["entry"].str.slice(8))
T = s1v.assign(actor=lab).groupby(["actor", "cendiv", "target"])["pv"].sum().unstack("target").reindex(columns=TARGETS)
T.to_csv(OUT / "cd_stage1.csv")
for a in ["electricity consumers", "owners (all groups)", "system", "readout", "health: AP2 ACS"]:
    print(a); display(T.loc[a].round(2))

electricity consumers


target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,2.26,2.87,14.42,18.47,45.00,78.38
East_South_Central,0.80,0.77,3.21,4.85,17.27,35.56
Mid_Atlantic,11.71,17.54,39.75,42.20,65.59,79.25
Mountain,5.23,9.33,15.00,18.85,35.21,62.86
New_England,2.51,3.45,7.81,7.05,14.12,16.67
Pacific,2.57,9.39,17.61,18.58,36.76,59.74
South_Atlantic,4.08,4.47,18.82,29.25,77.58,130.59
West_North_Central,0.84,1.59,10.52,14.00,31.58,50.91
West_South_Central,2.70,1.74,6.23,9.32,30.25,71.94


owners (all groups)


target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,-4.93,-3.82,-59.79,-63.82,-128.32,-183.26
East_South_Central,0.94,0.82,2.14,1.86,-4.65,-28.47
Mid_Atlantic,-20.03,-28.16,-53.87,-57.43,-74.94,-83.69
Mountain,-17.78,-28.09,-43.14,-45.46,-78.13,-111.99
New_England,-2.28,-3.77,-6.34,-5.69,-12.65,-14.82
Pacific,-2.66,-8.32,-18.98,-19.23,-36.67,-60.87
South_Atlantic,-11.66,-3.93,-48.65,-56.63,-137.78,-199.68
West_North_Central,-1.67,-2.52,-22.54,-28.53,-54.05,-73.54
West_South_Central,-8.70,-4.97,-10.58,-21.24,-43.66,-112.92


system


target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,-0.26,-2.44,-49.57,-50.96,-107.92,-133.11
East_South_Central,-0.12,0.56,3.33,3.69,8.33,-6.41
Mid_Atlantic,-14.88,-20.12,-28.40,-28.79,-21.73,-14.43
Mountain,-13.74,-17.58,-27.44,-27.92,-51.19,-65.45
New_England,4.57,2.83,10.02,10.67,12.03,16.79
Pacific,1.09,2.08,-1.71,0.65,-0.27,1.27
South_Atlantic,-11.33,1.63,-36.19,-38.34,-81.54,-103.41
West_North_Central,-0.63,-0.86,-14.41,-16.38,-24.88,-25.32
West_South_Central,-6.13,-3.96,-3.29,-11.50,-10.51,-36.40


readout


target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,1.63,1.74,12.20,13.45,24.63,37.76
East_South_Central,0.22,-0.06,2.02,3.91,16.04,29.32
Mid_Atlantic,0.99,1.11,3.67,4.26,7.25,12.82
Mountain,1.90,3.41,6.63,8.04,15.77,22.58
New_England,0.97,0.96,2.36,2.67,3.44,5.62
Pacific,2.30,3.22,5.73,5.80,9.05,13.13
South_Atlantic,0.74,0.89,6.45,9.62,24.81,46.55
West_North_Central,1.25,1.57,10.28,11.46,18.50,25.25
West_South_Central,2.89,1.80,3.77,9.82,18.02,36.07


health: AP2 ACS


target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,0.24,0.27,1.30,1.58,2.12,5.09
East_South_Central,-0.01,0.01,0.00,0.01,1.78,2.08
Mid_Atlantic,0.04,0.35,0.82,1.10,1.14,1.65
Mountain,0.04,0.07,0.13,0.15,0.30,0.44
New_England,0.02,0.03,0.03,0.04,0.04,0.05
Pacific,0.01,0.02,0.02,0.02,0.02,0.07
South_Atlantic,0.02,0.03,0.17,0.22,0.72,1.25
West_North_Central,0.02,0.06,0.25,0.36,0.74,1.41
West_South_Central,0.05,0.02,0.04,0.16,0.26,0.81


## 5. Stage-2 spread

Median and range across the runs of each stage, condition and target, per division and actor, at the
reference variant. Stage 2b's Δ contains the condition and the programme together.

In [8]:
r = PV[(PV["horizon"] == REF["horizon"]) & (PV["rate"] == REF["rate"]) & (PV["stage"] != "1")]
r = r.groupby(["stage", "condition", "target", "case", "cendiv", "actor"])["pv"].sum().reset_index()
SUM = r.groupby(["stage", "condition", "target", "cendiv", "actor"])["pv"].agg(["median", "min", "max", "size"]).reset_index()
SUM.to_csv(OUT / "cd_stage2_summary.csv", index=False)
x = SUM[(SUM["stage"] == "2a") & (SUM["actor"] == "electricity consumers")]
display(x.pivot_table(index="cendiv", columns="target", values="median").reindex(columns=TARGETS).round(2))

target,eia,aj,iaea,mck,cop28,eo
cendiv,,,,,,
East_North_Central,2.67,3.04,15.16,18.34,44.38,77.42
East_South_Central,1.07,0.75,3.21,4.52,16.91,35.03
Mid_Atlantic,13.67,22.45,39.78,44.72,65.21,78.99
Mountain,5.04,8.74,15.41,18.72,34.89,62.00
New_England,1.93,4.91,6.50,7.83,13.22,17.34
Pacific,3.33,8.89,15.20,18.30,36.32,60.83
South_Atlantic,4.21,4.95,19.59,28.52,76.83,131.80
West_North_Central,0.77,1.05,9.96,14.05,31.77,50.86
West_South_Central,2.47,1.53,8.37,9.17,30.90,69.03


## 6. What reads these exports

The H3 review page (notebook 25) and the observation log.